In [0]:
%sql
CREATE DATABASE IF NOT EXISTS delta_training_db;

In [0]:
%sql
use delta_training_db;

In [0]:
customers_data = [
    (1, "Aarav Sharma", "Hyderabad", "Gold", 25000),
    (2, "Priya Reddy", "Bengaluru", "Silver", 18000),
    (3, "Rohan Mehta", "Mumbai", "Gold", 32000),
    (4, "Sneha Iyer", "Chennai", "Bronze", 12000),
    (5, "Kiran Patel", "Hyderabad", "Silver", 22000),
    (6, "Ananya Das", "Kolkata", "Gold", 41000),
    (7, "Vikram Singh", "Delhi", "Bronze", 9000),
    (8, "Meera Nair", "Bengaluru", "Silver", 26000)
]

columns = ["customer_id", "customer_name", "city", "membership", "total_spend"]

customers_df = spark.createDataFrame(customers_data, columns)

display(customers_df)

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


In [0]:
%sql
CREATE OR REPLACE TABLE customer_delta_sql (
  customer_id int,
  customer_name string,
  city string,
  membership string,
  total_spend int
)
USING DELTA;

In [0]:
%sql
INSERT INTO customer_delta_sql VALUES
(1, "Aarav Sharma", "Hyderabad", "Gold", 25000),
    (2, "Priya Reddy", "Bengaluru", "Silver", 18000),
    (3, "Rohan Mehta", "Mumbai", "Gold", 32000);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM customer_delta_sql;

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000


In [0]:
customers_df.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("customer_delta_table")

In [0]:
%sql
SELECT * FROM customer_delta_table

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


In [0]:
%sql
CREATE OR REPLACE TABLE retail_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_orders VALUES

(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),

(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),

(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),

(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),

(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),

(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed');

num_affected_rows,num_inserted_rows
6,6


In [0]:

%sql
CREATE OR REPLACE TEMP VIEW new_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed')
AS new_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO retail_orders AS target
USING new_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status
WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)
VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
INSERT INTO retail_orders VALUES
(111,'Arjun Nair','Kerala','Laptop',2,75000,'Placed'),
(112,'Meera Singh','Delhi','Mobile',3,32000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
UPDATE retail_orders SET price=78000 WHERE product="Laptop"

num_affected_rows
3


In [0]:
%sql
DELETE FROM retail_orders WHERE quantity=1;

num_affected_rows
4


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',5,30000,'Delivered'),
(111,'Arjun Nair','Kerala','Laptop',2,75000,'Placed'),
(113,'Vikram Rao','Hyderabad','Tablet',1,24000,'Placed'),
(114,'Pooja Shah','Mumbai','Laptop',2,78000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO retail_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status
WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)
VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
-- INSERT Exercises
-- 1
INSERT INTO ecommerce_orders
VALUES (109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');


num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- 2
INSERT INTO ecommerce_orders VALUES
(110,'Arjun Nair','Kochi','Mobile',1,30000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- 3
INSERT INTO ecommerce_orders
VALUES (112,'Test User','Delhi','Camera',1,50000,'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- 4
INSERT INTO ecommerce_orders
VALUES (113,'Bulk Buyer','Pune','Headphones',5,2000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
-- 5
INSERT INTO ecommerce_orders VALUES
(114,'User1','Hyderabad','Mobile',1,25000,'Placed'),
(115,'User2','Hyderabad','Laptop',1,78000,'Placed'),
(116,'User3','Hyderabad','Tablet',2,26000,'Placed');

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
--PART 2 UPDATE exercises
-- 1
UPDATE ecommerce_orders
SET order_status = 'Shipped'
WHERE order_id = 101;



num_affected_rows
1


In [0]:
%sql
-- 2
UPDATE ecommerce_orders
SET quantity = quantity + 1
WHERE order_id = 102;



num_affected_rows
1


In [0]:
%sql
-- 3
UPDATE ecommerce_orders
SET price = 78000
WHERE product = 'Laptop';



In [0]:
%sql
-- 4
UPDATE ecommerce_orders
SET city = 'Secunderabad'
WHERE customer_name = 'Amit Sharma';



num_affected_rows
1


In [0]:
%sql
-- 5
UPDATE ecommerce_orders
SET order_status = 'Delivered'
WHERE product = 'Mobile';



num_affected_rows
4


In [0]:
%sql
-- 6
UPDATE ecommerce_orders
SET price = 2500
WHERE product = 'Headphones';



num_affected_rows
2


In [0]:
%sql
-- 7
UPDATE ecommerce_orders
SET price = price + 1000
WHERE product = 'Tablet';



num_affected_rows
3


In [0]:
%sql
-- 8
UPDATE ecommerce_orders
SET order_status = 'Processing'
WHERE city = 'Bangalore';



num_affected_rows
2


In [0]:
%sql
-- 9
UPDATE ecommerce_orders
SET quantity = 2
WHERE quantity = 1;



num_affected_rows
0


In [0]:
%sql
-- 10
UPDATE ecommerce_orders
SET city = 'Surat'
WHERE city = 'Ahmedabad';

num_affected_rows
0


In [0]:
%sql
--PART 3 DELETE exercises
-- 1
DELETE FROM ecommerce_orders
WHERE order_status = 'Cancelled';



num_affected_rows
1


In [0]:
%sql
-- 2
DELETE FROM ecommerce_orders
WHERE quantity > 3;



num_affected_rows
2


In [0]:
%sql
-- 3
DELETE FROM ecommerce_orders
WHERE product = 'Camera';



num_affected_rows
2


In [0]:
%sql
-- 4
DELETE FROM ecommerce_orders
WHERE city = 'Kolkata';



num_affected_rows
1


In [0]:
%sql
-- 5
DELETE FROM ecommerce_orders
WHERE price < 5000;



num_affected_rows
0


In [0]:
%sql
-- 6
DELETE FROM ecommerce_orders
WHERE customer_name LIKE 'A%';



num_affected_rows
0


In [0]:
%sql
-- 7
DELETE FROM ecommerce_orders
WHERE product = 'Tablet';

-- 8
DELETE FROM ecommerce_orders
WHERE city = 'Mumbai' AND quantity = 1;

-- 9
DELETE FROM ecommerce_orders
WHERE price > 80000;

-- 10 (last inserted)
DELETE FROM ecommerce_orders
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
%sql
-- 8
DELETE FROM ecommerce_orders
WHERE city = 'Mumbai' AND quantity = 1;



num_affected_rows
0


In [0]:
%sql
-- 9
DELETE FROM ecommerce_orders
WHERE price > 80000;



num_affected_rows
0


In [0]:
%sql
-- 10 (last inserted)
DELETE FROM ecommerce_orders
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
--PART 4 MERGE/UPSET
-- 1,2,3 Basic merge
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;

-- 4
SELECT * FROM ecommerce_orders ORDER BY order_id;

-- 5 (identify inserted vs updated)
SELECT t.order_id,
       CASE 
         WHEN s.order_id IS NULL THEN 'Existing'
         ELSE 'Inserted/Updated'
       END AS status
FROM ecommerce_orders t
LEFT JOIN incoming_orders s
ON t.order_id = s.order_id;

-- 6 (update only order_status)
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET t.order_status = s.order_status

WHEN NOT MATCHED THEN
  INSERT *;

-- 7 (update quantity only if increased)
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.quantity > t.quantity THEN
  UPDATE SET t.quantity = s.quantity

WHEN NOT MATCHED THEN
  INSERT *;

-- 8 (ignore cancelled orders)
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.order_status != 'Cancelled' THEN
  UPDATE SET *

WHEN NOT MATCHED AND s.order_status != 'Cancelled' THEN
  INSERT *;

-- 9 (insert only Laptop)
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN NOT MATCHED AND s.product = 'Laptop' THEN
  INSERT *;

-- 10 (update price only if higher)
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.price > t.price THEN
  UPDATE SET t.price = s.price

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
--PART 5  REAL UPSET
-- 1,2,3
MERGE INTO ecommerce_orders t
USING daily_updates s
ON t.order_id = s.order_id

WHEN MATCHED THEN
  UPDATE SET t.quantity = s.quantity

WHEN NOT MATCHED THEN
  INSERT *;

-- 4 (count updated)
SELECT COUNT(*) AS updated_rows
FROM ecommerce_orders t
JOIN daily_updates s
ON t.order_id = s.order_id;

-- 5 (count inserted)
SELECT COUNT(*) AS inserted_rows
FROM daily_updates s
LEFT JOIN ecommerce_orders t
ON t.order_id = s.order_id
WHERE t.order_id IS NULL;

-- 6
SELECT * FROM ecommerce_orders
ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
